# 06 — Iterative Burst Detector: Synthetic Validation

This notebook stress-tests `compute_iterative_bursts` against synthetic spike trains with known ground-truth burst intervals. The goal is to verify that:

- An independent-Poisson culture (the negative control) yields no/few network bursts.
- A cascade culture with discrete burst epochs yields one detection per ground-truth   burst — measured here as precision / recall / F1.

The synthetic-data primitives live in `yuxin_mea.analysis.synthetic_validation` (interval algebra, Poisson generators, GT evaluator). More elaborate generators (synchronous-onset, post-burst silence, long-wave oscillations, composite cultures) can be composed from those same primitives — the legacy `notebooks/06_*.ipynb` carries earlier prototypes that can be promoted to the package if they prove useful.

## Imports

`yuxin_mea` is an installable package (`pip install -e .` once). No `sys.path` hacks needed; imports work whether you launch Jupyter from the repo root or `notebooks/v2/`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from yuxin_mea.analysis import (
    IterativeBurstConfig,
    compute_iterative_bursts,
)
from yuxin_mea.analysis.synthetic_validation import (
    SyntheticDataset,
    generate_cascade_culture,
    generate_poisson_baseline,
    score_detection,
)

print('Imports OK')

## 1. Generate test datasets

Three cultures spanning the relevant detection regimes:

- **Poisson baseline** — independent units, no bursts. Negative control.
- **Cascade (sparse bursts)** — eight well-separated bursts. The easy case.
- **Cascade (dense bursts)** — twenty closely-spaced bursts with lower recruitment.   Probes detector behavior near the burstlet-merge gap and recruitment floor.

In [ ]:
DURATION_S = 60.0

datasets: list[SyntheticDataset] = [
    generate_poisson_baseline(
        n_units=32,
        duration_s=DURATION_S,
        rate_hz=1.0,
        seed=0,
    ),
    generate_cascade_culture(
        n_units=32,
        duration_s=DURATION_S,
        burst_centers_s=list(np.linspace(5.0, 55.0, 8)),
        burst_duration_s=0.5,
        burst_rate_hz=50.0,
        baseline_rate_hz=0.5,
        recruitment=0.9,
        seed=1,
    ),
    generate_cascade_culture(
        n_units=32,
        duration_s=DURATION_S,
        burst_centers_s=list(np.linspace(3.0, 57.0, 20)),
        burst_duration_s=0.4,
        burst_rate_hz=40.0,
        baseline_rate_hz=0.8,
        recruitment=0.6,
        seed=2,
    ),
]

summary = pd.DataFrame([
    {
        'name': ds.name,
        'culture_type': ds.culture_type,
        'n_units': len(ds.spike_times),
        'duration_s': ds.duration_s,
        'total_spikes': sum(len(v) for v in ds.spike_times.values()),
        'n_bursts_gt': len(ds.burst_intervals),
    }
    for ds in datasets
])
summary

## 2. Run the iterative burst detector

`compute_iterative_bursts` returns a `BurstResults` object; the column we score against is `result.network_bursts` (DataFrame with `start` / `end` in seconds). An empty DataFrame is the legitimate output for the Poisson baseline.

In [ ]:
config = IterativeBurstConfig()

detections: list[list[tuple[float, float]]] = []
results = []
for ds in datasets:
    try:
        result = compute_iterative_bursts(ds.spike_times, config=config)
    except Exception as exc:  # noqa: BLE001  -- want a row even if detection raises
        print(f'{ds.name}: detector raised {type(exc).__name__}: {exc}')
        detections.append([])
        results.append(None)
        continue

    nb = result.network_bursts
    if nb is None or len(nb) == 0:
        intervals: list[tuple[float, float]] = []
    else:
        intervals = [(float(s), float(e)) for s, e in zip(nb['start'], nb['end'])]
    detections.append(intervals)
    results.append(result)
    print(f'{ds.name}: detected {len(intervals)} network bursts')

## 3. Score detections

`score_detection` counts a detected interval as a true positive iff it overlaps some ground-truth interval by ≥ `min_overlap_s` AND that GT interval is not already claimed. Fragmented detections inflate FP rather than TP.

In [ ]:
rows = []
for ds, detected in zip(datasets, detections):
    scores = score_detection(detected, ds.burst_intervals, min_overlap_s=0.03)
    rows.append({
        'name': ds.name,
        'n_gt': len(ds.burst_intervals),
        'n_detected': len(detected),
        **scores,
    })

scores_df = pd.DataFrame(rows)
scores_df

## 4. Visualize one example with Plotly

Spike raster for the sparse-cascade dataset, with ground-truth bursts shaded red and detector outputs shaded blue. Where the bands overlap, the detector matched a real burst; isolated red is a missed burst (FN); isolated blue is a false positive.

In [ ]:
EXAMPLE_IDX = 1  # the sparse cascade culture
ds = datasets[EXAMPLE_IDX]
detected = detections[EXAMPLE_IDX]

unit_ids = list(ds.spike_times.keys())
unit_index = {uid: i for i, uid in enumerate(unit_ids)}

fig = go.Figure()

# Ground-truth burst windows (red) and detected windows (blue) as background shapes
shapes = []
for s, e in ds.burst_intervals:
    shapes.append(dict(
        type='rect', xref='x', yref='paper', x0=s, x1=e, y0=0, y1=1,
        fillcolor='red', opacity=0.18, line_width=0, layer='below',
    ))
for s, e in detected:
    shapes.append(dict(
        type='rect', xref='x', yref='paper', x0=s, x1=e, y0=0, y1=1,
        fillcolor='blue', opacity=0.18, line_width=0, layer='below',
    ))

# Spike raster: one scatter trace combining all units
xs: list[float] = []
ys: list[float] = []
for uid, times in ds.spike_times.items():
    if len(times) == 0:
        continue
    xs.extend(times.tolist())
    ys.extend([unit_index[uid]] * len(times))

fig.add_trace(go.Scattergl(
    x=xs, y=ys, mode='markers',
    marker=dict(size=2, color='black'),
    name='spikes',
))

# Dummy traces purely for legend entries
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(size=12, color='red', opacity=0.4),
    name='ground truth',
))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(size=12, color='blue', opacity=0.4),
    name='detected',
))

fig.update_layout(
    shapes=shapes,
    title=f'{ds.name} — GT (red) vs detected (blue)',
    xaxis_title='time (s)',
    yaxis_title='unit index',
    height=500,
    template='plotly_white',
)
fig

## 5. Extending the suite

More elaborate generators — synchronous-onset cultures, long-wave oscillations, post-burst silent periods, composite cultures with multiple burst morphologies — lived as inline code in the legacy `notebooks/06_iterative_burst_detector_synthetic_validation.ipynb`. If any of them prove useful for ongoing detector validation, promote them into `yuxin_mea.analysis.synthetic_validation` (and add unit tests) so they can be reused from this notebook and from the test suite.